# ① 전처리 — 문서 파싱 · 청크 분할
담당: 전처리 · 코드: `rag/preprocessing/parsing.py`, `rag/preprocessing/chunking.py`

`config.yaml` 의 `paths.input`(ZIP 또는 폴더)에 있는 PDF · HWP · HWPX 를 읽어 본문 · 표 · (선택) 그림 설명을 뽑고, 청크로 나눕니다.

| 저장 파일 | 읽는 노트북 |
|---|---|
| `artifacts/chunks.jsonl` (청크) | 02 검색, 03 검증 |
| `artifacts/docs.jsonl` (문서 목록 — 발주기관 라우팅용) | 02 검색 |

In [ ]:
from rag.config import ARTIFACTS, ROOT, load_config
from rag.preprocessing.chunking import chunk_documents, chunk_summary, load_v11_package, save_chunks
from rag.preprocessing.parsing import collect_inputs, parse_documents, parsing_summary
from rag.usage import usage_summary

cfg = load_config()

## 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
`OCR = True`이면 선택한 비전 모델로 그림을 텍스트로 바꿉니다(OpenAI API, 유료). 기본값은 `False`이며, 이때 그림을 API로 보내지 않습니다.
청킹 크기는 본문 글자 수 기준이며, 변경 후 01부터 후속 노트북을 다시 실행해야 합니다.
`PREPROCESSED = None`이면 현재 파서를 사용합니다. v11 패키지를 쓰려면 `"data/RFP_Preprocessing_Final_v11_Team"`으로 바꾸세요.
패키지 모드에서는 저장된 OCR 텍스트를 그대로 읽으며 파싱·OCR·청크 재분할을 건너뜁니다.


In [ ]:
PREPROCESSED = None  # v11 사용 시 "data/RFP_Preprocessing_Final_v11_Team"
OCR = False
OCR_MODEL = cfg["ocr"]["model"]                    # 예: "gpt-5-mini"
OCR_REASONING_EFFORT = cfg["ocr"]["reasoning_effort"]  # 예: "low"
OCR_MAX_TOKENS = cfg["ocr"]["max_tokens"]          # 출력·추론 토큰 한도
OCR_PRICES = cfg["ocr"]["usd_per_1m_tokens"]     # 입력 · 캐시 입력 · 출력 단가
CHUNK_SIZE = cfg["chunking"]["chunk_size"]         # 예: 700

cfg["ocr"].update(model=OCR_MODEL, reasoning_effort=OCR_REASONING_EFFORT, max_tokens=OCR_MAX_TOKENS,
                  usd_per_1m_tokens=OCR_PRICES)
cfg["chunking"]["chunk_size"] = CHUNK_SIZE
cfg["ocr"], cfg["chunking"]


## 1. 입력 파일 모으기
ZIP(안의 ZIP 포함)을 `artifacts/raw/` 에 풀고 PDF · HWP · HWPX 목록을 만듭니다.

In [ ]:
if PREPROCESSED is None:
    files = collect_inputs(ROOT / cfg["paths"]["input"], ARTIFACTS / "raw")
    print(f"입력 파일: {len(files)}")

## 2. 파싱
문서마다 쪽 · 표 · 그림 수와 표지 감지 결과를 봅니다.

In [ ]:
if PREPROCESSED is None:
    docs = parse_documents(files, cfg["parsing"], ocr=OCR, ocr_cfg=cfg["ocr"])
    display(parsing_summary(docs))
else:
    docs, chunks = load_v11_package(ROOT / PREPROCESSED)
    print(f"v11 문서: {len(docs)}, 청크: {len(chunks)}")

## 3. 청크 분할 · 저장

In [ ]:
if PREPROCESSED is None:
    chunks = chunk_documents(docs, cfg["chunking"])
save_chunks(docs, chunks, ARTIFACTS)
chunk_summary(chunks)

## API 사용량
호출마다 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 표는 누적 사용량이며 OCR을 끄면 새 호출이 없습니다.

In [ ]:
usage_summary("ocr")